In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import polars as pl

from config import (
    RAW_DATA_DIR,
    SAML_D_DATASET_HANDLE,
    SAML_D_FILE_NAME
)

from src.data.acquisition import KaggleDatasetDownloader
from src.data.loader import SAMLDDataLoader
from src.data.validation import SAMLDDataValidator

In [2]:
### data acquisition
# create a downloader instance
downloader = KaggleDatasetDownloader(
    dataset_handle = SAML_D_DATASET_HANDLE,
    file_name = SAML_D_FILE_NAME,
    destination_dir = RAW_DATA_DIR
)

# download the raw SAML-D dataset
raw_data_path = downloader.download(
    force = False
)

raw_data_path

100%|██████████| 193M/193M [00:13<00:00, 14.6MB/s] 

Extracting zip of SAML-D.csv...


PosixPath('/Users/gorkemy/Desktop/Projects/Graph-based Financial Crime/data/raw/SAML-D.csv')

In [3]:
### data inspection
# create a loader instance
loader = SAMLDDataLoader(
    raw_data_path = raw_data_path
)

# create lazy query plan
raw_lazy_frame = loader.scan_raw()

# display LazyFrame schema
display(raw_lazy_frame.collect_schema())

# load a LazyFrame sample
raw_sample = loader.load_raw_sample(
    n_rows = 10_000
)

# inspect first 10 rows
display(raw_sample.head(10))

# inspecting the primary label
display(
    raw_sample
    .group_by('Is_laundering')
    .len()
)

Schema([('Time', String),
        ('Date', String),
        ('Sender_account', String),
        ('Receiver_account', String),
        ('Amount', Float64),
        ('Payment_currency', String),
        ('Received_currency', String),
        ('Sender_bank_location', String),
        ('Receiver_bank_location', String),
        ('Payment_type', String),
        ('Is_laundering', Int8),
        ('Laundering_type', String)])

Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type
str,str,str,str,f64,str,str,str,str,str,i8,str
"""10:35:19""","""2022-10-07""","""8724731955""","""2769355426""",1459.15,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Cash Deposit""",0,"""Normal_Cash_Deposits"""
"""10:35:20""","""2022-10-07""","""1491989064""","""8401255335""",6019.64,"""UK pounds""","""Dirham""","""UK""","""UAE""","""Cross-border""",0,"""Normal_Fan_Out"""
"""10:35:20""","""2022-10-07""","""287305149""","""4404767002""",14328.44,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Cheque""",0,"""Normal_Small_Fan_Out"""
"""10:35:21""","""2022-10-07""","""5376652437""","""9600420220""",11895.0,"""UK pounds""","""UK pounds""","""UK""","""UK""","""ACH""",0,"""Normal_Fan_In"""
"""10:35:21""","""2022-10-07""","""9614186178""","""3803336972""",115.25,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Cash Deposit""",0,"""Normal_Cash_Deposits"""
"""10:35:21""","""2022-10-07""","""8974559268""","""3143547511""",5130.99,"""UK pounds""","""UK pounds""","""UK""","""UK""","""ACH""",0,"""Normal_Group"""
"""10:35:23""","""2022-10-07""","""980191499""","""8577635959""",12176.52,"""UK pounds""","""UK pounds""","""UK""","""UK""","""ACH""",0,"""Normal_Small_Fan_Out"""
"""10:35:23""","""2022-10-07""","""8057793308""","""9350896213""",56.9,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Credit card""",0,"""Normal_Small_Fan_Out"""
"""10:35:26""","""2022-10-07""","""6116657264""","""656192169""",4738.45,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Cheque""",0,"""Normal_Fan_Out"""


Is_laundering,len
i8,u32
0,9987
1,13


In [4]:
### data validation
# raw data content validation
raw_content_summary = SAMLDDataValidator.validate_content(
    lazy_frame = raw_lazy_frame
)

raw_content_summary_frame = pl.DataFrame(
    [raw_content_summary]
)

with pl.Config(
    tbl_rows=-1
):
    display(
    raw_content_summary_frame
    .transpose(
        include_header = True,
        header_name = 'metric',
        column_names = ['value']
    )
)

metric,value
str,i64
"""row_count""",9504852
"""time_null_count""",0
"""date_null_count""",0
"""sender_account_null_count""",0
"""receiver_account_null_count""",0
"""amount_null_count""",0
"""payment_currency_null_count""",0
"""received_currency_null_count""",0
"""sender_bank_location_null_coun…",0
